In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:

!pip -q install ultralytics opencv-python tqdm
import os,json,cv2,numpy as np
from tqdm import tqdm



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.6 MB/s eta 0:00:00


In [ ]:
# I changed these input/output directories each run based on dataset version
INPUT_ROOT="/content/drive/MyDrive/chronocounter/dataset v1.0"; OUT_ROOT="/content/drive/MyDrive/chronocounter/dataset v1.0 sliced"

# This is all AI-generated code for slicing the dataset into smaller overlapping 640x640 images the model can train on
# Also converts the dataset from COCO 1.0 into YOLO format
TW=TH=640; OVER=0.2; MIN_BOX=4
os.makedirs(f"{OUT_ROOT}/images/train",exist_ok=True);os.makedirs(f"{OUT_ROOT}/labels/train",exist_ok=True)
coco=json.load(open(os.path.join(INPUT_ROOT,"annotations","instances_default.json")))
imgs={i["id"]:i for i in coco["images"]}; anns={}
for a in coco["annotations"]:
    if a.get("iscrowd",0): continue
    anns.setdefault(a["image_id"],[]).append(a)
cats=sorted(coco["categories"],key=lambda c:c["id"]); cmap={c["id"]:i for i,c in enumerate(cats)}
def positions(L,T,O):
    if L<=T:return [0]
    step=max(1,int(T*(1-O)));p=[];x=0
    while x+T<L:
        p.append(x);x+=step
    if p[-1]!=L-T:p.append(L-T)
    return p
for iid,imeta in tqdm(imgs.items()):
    img=cv2.imread(os.path.join(INPUT_ROOT,"images","train",imeta["file_name"]))
    print(imeta["file_name"])
    h,w=img.shape[:2]
    for y in positions(h,TH,OVER):
      for x in positions(w,TW,OVER):
        crop=img[y:min(y+TH,h),x:min(x+TW,w)]
        tile=np.zeros((TH,TW,3),dtype=img.dtype);tile[:crop.shape[0],:crop.shape[1]]=crop
        stem=os.path.splitext(imeta["file_name"])[0]+f"_{y}_{x}"
        cv2.imwrite(f"{OUT_ROOT}/images/train/{stem}.png",tile)
        out=[]
        for a in anns.get(iid, []):
          bx, by, bw, bh = a["bbox"]

          # Original box coordinates
          orig_x1 = bx
          orig_y1 = by
          orig_x2 = bx + bw
          orig_y2 = by + bh

          # Compute intersection with tile
          clip_x1 = max(orig_x1, x)
          clip_y1 = max(orig_y1, y)
          clip_x2 = min(orig_x2, x + TW)
          clip_y2 = min(orig_y2, y + TH)

          # No overlap
          if clip_x2 <= clip_x1 or clip_y2 <= clip_y1:
              continue

          clipped_w = clip_x2 - clip_x1
          clipped_h = clip_y2 - clip_y1

          # Remove tiny boxes
          if clipped_w < MIN_BOX or clipped_h < MIN_BOX:
              continue

          # -------- Retained-area test --------

          original_area = bw * bh
          clipped_area = clipped_w * clipped_h

          retained_fraction = clipped_area / original_area

          keep_box = False

          # Special handling for category ID 3 ("long rest")
          if a["category_id"] == 3:

              # Rule 1: keep if at least 50% of the full box remains
              if retained_fraction >= 0.5:
                  keep_box = True
              else:
                  # Rule 2: check whether at least 50% of the middle third remains

                  # Middle third of the original bounding box
                  mid_x1 = bx + bw / 3.0
                  mid_x2 = bx + 2.0 * bw / 3.0
                  mid_y1 = by
                  mid_y2 = by + bh

                  # Intersection of the tile with the middle region
                  mid_clip_x1 = max(mid_x1, x)
                  mid_clip_y1 = max(mid_y1, y)
                  mid_clip_x2 = min(mid_x2, x + TW)
                  mid_clip_y2 = min(mid_y2, y + TH)

                  if mid_clip_x2 > mid_clip_x1 and mid_clip_y2 > mid_clip_y1:
                      middle_area = (mid_x2 - mid_x1) * (mid_y2 - mid_y1)
                      retained_middle_area = (
                          (mid_clip_x2 - mid_clip_x1)
                          * (mid_clip_y2 - mid_clip_y1)
                      )

                      if retained_middle_area / middle_area >= 0.5:
                          keep_box = True

          # All other categories use the standard rule
          else:
              if retained_fraction >= 0.5:
                  keep_box = True

          if not keep_box:
              continue

          # --------------------------------------------

          cx = ((clip_x1 + clip_x2) / 2 - x) / TW
          cy = ((clip_y1 + clip_y2) / 2 - y) / TH

          nw = clipped_w / TW
          nh = clipped_h / TH

          out.append(
              f"{cmap[a['category_id']]} "
              f"{cx:.6f} {cy:.6f} "
              f"{nw:.6f} {nh:.6f}"
          )
        open(f"{OUT_ROOT}/labels/train/{stem}.txt","w").write("\n".join(out))
open(f"{OUT_ROOT}/data.yaml","w").write(f"path: {OUT_ROOT}\ntrain: images/train\nval: images/train\nnc: {len(cats)}\nnames: {[c['name'] for c in cats]}\n")

  0%|          | 0/55 [00:00<?, ?it/s]

IMSLP299526-PMLP03667-Ravel-Bolero_Picc-1.png


  2%|▏         | 1/55 [00:00<00:30,  1.78it/s]

IMSLP299526-PMLP03667-Ravel-Bolero_Picc-2.png


  4%|▎         | 2/55 [00:01<00:29,  1.80it/s]

IMSLP299526-PMLP03667-Ravel-Bolero_Picc-3.png


... (102 outputs left out)

 98%|█████████▊| 54/55 [01:34<00:02,  2.95s/it]

scanned_dataset_v1.0_9.png


100%|██████████| 55/55 [01:37<00:00,  1.77s/it]


221